# 1 Tokenization

In [1]:
# Simple Word and Sentence Tokenization (Without NLTK)

# Taking input
text = input("Enter a paragraph: ")

# Sentence Tokenization
sentences = text.split(".")

print("\nSentences:")
for s in sentences:
    if s.strip() != "":
        print(s.strip())

# Word Tokenization
words = text.split()

print("\nWords:")
for w in words:
    print(w)

Enter a paragraph:  Natural Language Processing is interesting. It is part of AI.



Sentences:
Natural Language Processing is interesting
It is part of AI

Words:
Natural
Language
Processing
is
interesting.
It
is
part
of
AI.


# 2 Porter stemmer algo

In [2]:
# Simple Porter Stemmer Implementation (Basic Version - Without NLTK)

def porter_stemmer(word):
    if word.endswith("ing"):
        return word[:-3]
    elif word.endswith("ed"):
        return word[:-2]
    elif word.endswith("ly"):
        return word[:-2]
    elif word.endswith("s") and len(word) > 1:
        return word[:-1]
    else:
        return word

# Taking input sentence
text = input("Enter a sentence: ")

words = text.split()

print("\nStemmed Words:")
for word in words:
    print(word, "->", porter_stemmer(word))

Enter a sentence:  playing jumped happily cats



Stemmed Words:
playing -> play
jumped -> jump
happily -> happi
cats -> cat


# 3 Sentence Boundary (Using ML)

In [5]:
# Sentence Boundary Detection using ML (Working Version)

import re
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

# Training Data (manually created small dataset)
training_data = [
    ({"prev_word": "world", "next_cap": True}, 1),
    ({"prev_word": "Dr", "next_cap": True}, 0),
    ({"prev_word": "raining", "next_cap": True}, 1),
    ({"prev_word": "U", "next_cap": False}, 0),
]

X_train = [features for features, label in training_data]
y_train = [label for features, label in training_data]

vec = DictVectorizer()
X_train_vec = vec.fit_transform(X_train)

model = LogisticRegression()
model.fit(X_train_vec, y_train)

# ===== USER INPUT =====
text = input("Enter text: ")

# Find all dots
matches = list(re.finditer(r'\.', text))

print("\nPredictions:")

for match in matches:
    index = match.start()
    
    prev_word = text[:index].split()[-1]
    next_part = text[index+1:].strip()
    
    if next_part:
        next_cap = next_part[0].isupper()
    else:
        next_cap = False
    
    features = {"prev_word": prev_word, "next_cap": next_cap}
    X_test = vec.transform([features])
    prediction = model.predict(X_test)[0]
    
    print(f"Dot after '{prev_word}' ->",
          "Sentence Boundary" if prediction == 1 else "Not Boundary")

Enter text:  Dr. Smith is here. It is raining.



Predictions:
Dot after 'Dr' -> Not Boundary
Dot after 'here' -> Sentence Boundary
Dot after 'raining' -> Sentence Boundary


# 4 Boundary Detection (Using ML)

In [6]:
# Topic Boundary Detection using ML

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Training Data (Very small example dataset)
train_paragraphs = [
    "Machine learning is part of artificial intelligence",
    "Neural networks are used in deep learning",
    "Cricket is a popular sport in India",
    "Football is played worldwide"
]

labels = [0, 0, 1, 1]  # 0 = AI topic, 1 = Sports topic

# Convert text to numerical form
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_paragraphs)

# Train model
model = LogisticRegression()
model.fit(X_train, labels)

# ===== USER INPUT =====
text = input("Enter a paragraph: ")

X_test = vectorizer.transform([text])
prediction = model.predict(X_test)

if prediction[0] == 0:
    print("Topic: Artificial Intelligence")
else:
    print("Topic: Sports")

Enter a paragraph:  Deep learning uses neural networks


Topic: Artificial Intelligence


# 5 phrase parsing using nltk

In [7]:
# Phrase Structure Tree using NLTK

import nltk
from nltk import CFG
from nltk.parse import ChartParser

# Define Grammar
grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | N
VP -> V NP
Det -> 'the'
N -> 'cat' | 'dog'
V -> 'chased' | 'saw'
""")

parser = ChartParser(grammar)

# Take input
sentence = input("Enter sentence (use words: the cat dog chased saw): ")
words = sentence.split()

# Parse
for tree in parser.parse(words):
    print(tree)
    tree.pretty_print()

Enter sentence (use words: the cat dog chased saw):  the cat chased the dog


(S (NP (Det the) (N cat)) (VP (V chased) (NP (Det the) (N dog))))
              S               
      ________|_____           
     |              VP        
     |         _____|___       
     NP       |         NP    
  ___|___     |      ___|___   
Det      N    V    Det      N 
 |       |    |     |       |  
the     cat chased the     dog



# 6 Dependency tree using nltk

In [9]:
# Simple Dependency Tree using NLTK

import nltk
from nltk.tree import Tree

# Input sentence
sentence = input("Enter sentence (example: Ram eats mango): ")

words = sentence.split()

# Manual dependency structure
# Assuming format: Subject Verb Object

if len(words) == 3:
    root = words[1]
    tree = Tree(root, [words[0], words[2]])
    print("\nDependency Tree:")
    tree.pretty_print()
else:
    print("Please enter 3-word sentence (e.g., Ram eats mango)")

Enter sentence (example: Ram eats mango):  Ram eats mango



Dependency Tree:
    eats      
  ___|_____    
Ram      mango



# 7 SR Parsing 

In [1]:
# Shift Reduce Parser (Short Version with Table + Tree)

rules = {
    ("id",): "S",
    ("S", "+", "S"): "S"
}

# tokenize
s = input("Enter expression (example: id+id): ")
tokens, i = [], 0
while i < len(s):
    if s[i:i+2] == "id":
        tokens.append("id"); i += 2
    else:
        tokens.append(s[i]); i += 1

# node for parse tree
class N:
    def __init__(self, v, c=[]): self.v, self.c = v, c

def show(n, l=0):
    print("  "*l + n.v)
    for ch in n.c: show(ch, l+1)

stack, tstack, steps = [], [], []

print("\nStack\t\tInput\t\tAction")

while True:
    if tokens:  # SHIFT
        x = tokens.pop(0)
        stack.append(x)
        tstack.append(N(x))
        steps.append((stack.copy(), tokens.copy(), "Shift"))
        print(stack, "\t", tokens, "\t Shift")

    changed = True
    while changed:  # REDUCE
        changed = False
        for rhs, lhs in rules.items():
            if tuple(stack[-len(rhs):]) == rhs:
                ch = tstack[-len(rhs):]
                stack, tstack = stack[:-len(rhs)], tstack[:-len(rhs)]
                stack.append(lhs)
                tstack.append(N(lhs, ch))
                steps.append((stack.copy(), tokens.copy(), "Reduce " + "".join(rhs)))
                print(stack, "\t", tokens, "\t Reduce", "".join(rhs))
                changed = True

    if not tokens: break

# result
if stack == ["S"]:
    print("\nString Accepted\n\nParse Tree:")
    show(tstack[0])
else:
    print("\nString Rejected")

print("\nParsing Table:")
for s in steps:
    print(s[0], "\t", s[1], "\t", s[2])

Enter expression (example: id+id):  id+id



Stack		Input		Action
['id'] 	 ['+', 'id'] 	 Shift
['S'] 	 ['+', 'id'] 	 Reduce id
['S', '+'] 	 ['id'] 	 Shift
['S', '+', 'id'] 	 [] 	 Shift
['S', '+', 'S'] 	 [] 	 Reduce id
['S'] 	 [] 	 Reduce S+S

String Accepted

Parse Tree:
S
  S
    id
  +
  S
    id

Parsing Table:
['id'] 	 ['+', 'id'] 	 Shift
['S'] 	 ['+', 'id'] 	 Reduce id
['S', '+'] 	 ['id'] 	 Shift
['S', '+', 'id'] 	 [] 	 Shift
['S', '+', 'S'] 	 [] 	 Reduce id
['S'] 	 [] 	 Reduce S+S


# 8 Morphological Models

In [3]:
# -------- INPUT --------
word = input("Enter word: ")

# -------- 1. DICTIONARY LOOKUP --------
def dict_lookup(w):
    lexicon = {
        "cats": ("cat", "NOUN+PL"),
        "running": ("run", "VERB+PROG"),
        "went": ("go", "VERB+PAST")
    }
    return lexicon.get(w, ("Unknown", "Unknown"))

# -------- 2. FINITE STATE (rule-based) --------
def fsm(w):
    if w.endswith("s"):
        return (w[:-1], "NOUN+PL")
    elif w.endswith("ing"):
        return (w[:-3], "VERB+PROG")
    elif w.endswith("ed"):
        return (w[:-2], "VERB+PAST")
    return ("Unknown", "Unknown")

# -------- 3. UNIFICATION BASED --------
def unify(w):
    root, feat = fsm(w)  # reuse rules
    if root != "Unknown":
        return (root, {"pos": feat.split("+")[0], "tense/num": feat.split("+")[1]})
    return ("Unknown", {})

# -------- OUTPUT --------
print("\nDictionary Lookup:", dict_lookup(word))
print("Finite State Model:", fsm(word))
print("Unification Model:", unify(word))

Enter word:  walking



Dictionary Lookup: ('Unknown', 'Unknown')
Finite State Model: ('walk', 'VERB+PROG')
Unification Model: ('walk', {'pos': 'VERB', 'tense/num': 'PROG'})


# 9 WSD Algorithm

In [4]:
# -------- INPUT --------
sentence = input("Enter sentence: ").lower()
target = input("Enter ambiguous word: ").lower()

# -------- SENSE DEFINITIONS (knowledge base) --------
kb = {
    "bank": {
        "finance": "money deposit loan account finance",
        "river": "water river shore stream flood"
    },
    "bat": {
        "animal": "nocturnal flying mammal cave",
        "sports": "cricket baseball hit ball"
    }
}

# -------- WSD (Lesk-style overlap) --------
def wsd(sent, word):
    words = set(sent.split())
    senses = kb.get(word, {})
    best_sense, max_overlap = None, 0
    
    for sense, desc in senses.items():
        overlap = len(words & set(desc.split()))
        if overlap > max_overlap:
            best_sense, max_overlap = sense, overlap
    
    return best_sense if best_sense else "Unknown"

# -------- OUTPUT --------
print("Predicted Sense:", wsd(sentence, target))

Enter sentence:  I deposited money in the bank
Enter ambiguous word:  bank


Predicted Sense: finance


# 10 Supervised ML WSD algorithm

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# -------- TRAINING DATA --------
data = [
    ("I deposited money in the bank", "finance"),
    ("She works at a bank", "finance"),
    ("The river bank is beautiful", "river"),
    ("He sat near the bank of the river", "river")
]

X = [d[0] for d in data]
y = [d[1] for d in data]

# -------- TRAIN MODEL --------
vec = CountVectorizer()
X_vec = vec.fit_transform(X)

model = MultinomialNB()
model.fit(X_vec, y)

# -------- INPUT --------
sentence = input("Enter sentence: ")

# -------- PREDICT --------
pred = model.predict(vec.transform([sentence]))

print("Predicted Sense:", pred[0])

11. Predicate Argument Structure

In [1]:
sentence = input("Enter a sentence: ")

words = sentence.split()

if len(words) < 2:
    print("Not enough words to form predicate-argument structure")
else:
    subject = words[0]
    predicate = words[1]
    obj = " ".join(words[2:]) if len(words) > 2 else "None"

    print("\nPredicate-Argument Structure:")
    print("Predicate:", predicate)
    print("Argument 1 (Subject):", subject)
    print("Argument 2 (Object):", obj)

Enter a sentence:  ram eats mango



Predicate-Argument Structure:
Predicate: eats
Argument 1 (Subject): ram
Argument 2 (Object): mango
